In [ ]:
# start coding here

# Add Residual Loads for Austria
Provided residual loads will replace the austrian residual loads generated by PyPSA-Eur.

## Load files as dataframes

In [ ]:
snakemake.input.loads_dir

In [ ]:
import pandas as pd

df = pd.read_csv(f"{snakemake.input.loads_dir}/biomasse.csv", index_col=0, parse_dates=True, delimiter=";")
df


In [ ]:
df

In [ ]:
df = pd.read_csv(f"{snakemake.input.loads_dir}/GrossWasserkraft.csv", index_col="Zeitpunkt", parse_dates=True, delimiter=";")
df.head

In [ ]:
df.sum().describe()


In [ ]:
export = pd.read_csv(f"{snakemake.input.loads_dir}/andere/Export.csv", index_col=0, parse_dates=True, delimiter=";")
export.head

## Residuallasten aufsummieren
Residuallasten liegen in den CSV Dateien als normierte Einheiten vor. Daher werden die Zeitreihen nach Knoten summiert und mit dem Gesamtverbrauch von Österreich multipliziert, um die Einheit MW zu bekommen.
Danach wird die summierte Zeitreihe wieder auf die Österreichischen Netzknoten aufgeteilt.

In [ ]:
import pandas as pd
from pathlib import Path

# Ordner mit Zeitreihen
folder = Path(f"{snakemake.input.loads_dir}/NIP2030")  # oder NIP2040

# Nur CSV-Dateien (z. B. 'Nachfrage.csv', 'PV.csv', 'Wind.csv', 'Export.csv' etc.)
data_files = [f for f in folder.glob("*.csv") if f.name != "Metadata.csv"]

# Initialisiere leeres DataFrame für Gesamtsumme
total_relative_profile = None

for f in data_files:
    df = pd.read_csv(f, index_col=0, delimiter=";", parse_dates=True)
    
    # Bei Export.csv liegt das Zeitprofil ab Zeile 1, Zeile 0 ist Mapping
    if f.name.lower().startswith("export"):
        df = df.drop(index=df.index[0])  # Mapping-Zeile entfernen

    df = df.astype(float)

    summed = df.sum(axis=1)  # Zeitreihe: Summe aller Spalten für jede Stunde
    # Index zurücksetzen (damit jeder summierte Zeitreihe eindeutig ist)
    summed.index = pd.RangeIndex(start=0, stop=len(summed))

    if total_relative_profile is None:
        total_relative_profile = summed
    else:
        total_relative_profile += summed


In [ ]:
# Gesamtverbrauch als Zeitreihe (z. B. aus PyPSA oder Statistik Austria)
real_total_load_mw = 5500  # Beispiel: ∅ Verbrauch in MW
residual_load_mw = total_relative_profile / total_relative_profile.mean() * real_total_load_mw


In [ ]:
total_relative_profile.mean()

In [ ]:
residual_load_mw

In [ ]:
# Beispiel: Verteilung anhand bestehender Buslast im PyPSA-Netz
load_share = network.loads_t.p_set.iloc[0] / network.loads_t.p_set.iloc[0].sum()

# Multipliziere jede Stunde mit Verteilung
distributed_residual = residual_load_mw[:, None] * load_share.values[None, :]

# Als DataFrame mit passenden Busnamen
distributed_df = pd.DataFrame(distributed_residual, index=residual_load_mw.index, columns=network.loads_t.p_set.columns)


## Import Real Austrian Load-Series
CSV data exported from entsoe transparency portal for AT-2024.
Data is 15-minute based load series for 1 year.

In [ ]:
import pandas as pd

# ENTSO-E Datei laden
entsoe_path = "data/MA/loads/entsoeAT/Total Load - Day Ahead _ Actual_202401010000-202501010000.csv"
entsoe_df = pd.read_csv(entsoe_path, skiprows=0)

# Erste Zeilen ansehen
entsoe_df.head()
